# Step 5 — Support Strategy Agent
**Phase 3 | NIKKO Engineering Collective**

Tests `agents/support_strategy_agent.py`. No model required for Parts A.

**What we are testing:**
1. crisis_bypass() returns the static crisis strategy without LLM call
2. strategize() raises ValueError when called with mode=CRISIS
3. mock_strategize() round-trips a valid payload correctly
4. mode and distress_level are always taken from Router/Signal (not model output)
5. Safe fallback triggers on empty fields, with mode-appropriate defaults
6. JSON extraction handles markdown fences and preamble


In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from agents.support_strategy_agent import SupportStrategyAgent
from agents.router import Router
from agents.signal_agent import SignalAgent
from docs.schemas.acp_schemas import DistressLevel, OperationalMode

ssa = SupportStrategyAgent()   # no model load
router = Router()
sa = SignalAgent()
print('Agents instantiated.')

Agents instantiated.


In [2]:
def make_signal(distress='moderate', risk=None, support=None, behavioral=None, confidence=0.75):
    return sa.mock_analyze({
        'distress_level': distress,
        'emotional_states': ['sadness_spectrum.low_mood_language'],
        'cognitive_patterns': [],
        'behavioral_indicators': behavioral or [],
        'risk_indicators': risk or [],
        'support_needs': support or ['emotional_validation'],
        'confidence': confidence,
        'uncertainty_notes': ''
    })

print('Helper ready.')

Helper ready.


## Part A — Logic tests (no model required)


In [3]:
# --- T1: crisis_bypass() returns static strategy without LLM call ---
strategy = ssa.crisis_bypass()
ok = (
    strategy.mode == OperationalMode.CRISIS
    and strategy.distress_level == DistressLevel.CRISIS
    and len(strategy.response_constraints) >= 4
    and 'Lifeline' in ' '.join(strategy.response_constraints)
)
print(f'T1 crisis_bypass() returns static strategy : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={strategy.mode.value}  constraints={len(strategy.response_constraints)}')
print(f'   tone: {strategy.tone_guidance[:70]!r}')

T1 crisis_bypass() returns static strategy : ✓
   mode=crisis  constraints=6
   tone: 'Calm, grounded, and human. Do not minimise the severity. Do not panic.'


In [4]:
# --- T2: strategize() raises ValueError when called with mode=CRISIS ---
signal = make_signal(distress='crisis', risk=['risk.active.suicidal_ideation'])
crisis_decision = router.route(signal)

try:
    ssa.strategize(crisis_decision, signal)
    print(f'T2 CRISIS guard : {chr(10007)} FAIL — should have raised ValueError')
except ValueError as e:
    print(f'T2 CRISIS guard raises ValueError : {chr(10003)}')
    print(f'   {str(e)[:80]}')

T2 CRISIS guard raises ValueError : ✓
   strategize() must not be called in Crisis Mode. Use crisis_bypass() instead. (ag


In [5]:
# --- T3: mock_strategize() COMFORT round-trip ---
signal = make_signal(distress='moderate', support=['emotional_validation'], confidence=0.72)
comfort_decision = router.route(signal)

mock = {
    'tone_guidance': 'Warm, present, and validating. Mirror the emotional weight without amplifying it.',
    'framing_strategy': 'Lead with acknowledgement of the user feelings before anything else.',
    'response_constraints': ['No advice unless explicitly requested.', 'No solution-framing.', 'Keep responses short and human.']
}

strategy = ssa.mock_strategize(mock, comfort_decision, signal)
# mode and distress_level must come from Router/Signal, NOT from the mock dict
ok = (
    strategy.mode == OperationalMode.COMFORT
    and strategy.distress_level == DistressLevel.MODERATE
    and strategy.tone_guidance == mock['tone_guidance']
    and len(strategy.response_constraints) == 3
)
print(f'T3 COMFORT mock round-trip : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={strategy.mode.value}  distress={strategy.distress_level.value}')
print(f'   tone: {strategy.tone_guidance[:70]!r}')

T3 COMFORT mock round-trip : ✓
   mode=comfort  distress=moderate
   tone: 'Warm, present, and validating. Mirror the emotional weight without amp'


In [6]:
# --- T4: mock_strategize() GUIDANCE round-trip ---
signal = make_signal(distress='low', support=['psychoeducation', 'emotional_validation'], confidence=0.80)
guidance_decision = router.route(signal)

mock = {
    'tone_guidance': 'Calm, informative, grounded. Epistemic humility throughout.',
    'framing_strategy': 'Offer information as something others have found helpful, not as prescription.',
    'response_constraints': ['Evidence must be cited.', 'No directive advice.', 'Autonomy language required.']
}

strategy = ssa.mock_strategize(mock, guidance_decision, signal)
ok = (
    strategy.mode == OperationalMode.GUIDANCE
    and strategy.distress_level == DistressLevel.LOW
    and 'cited' in ' '.join(strategy.response_constraints)
)
print(f'T4 GUIDANCE mock round-trip : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={strategy.mode.value}  distress={strategy.distress_level.value}')

T4 GUIDANCE mock round-trip : ✓
   mode=guidance  distress=low


In [7]:
# --- T5: mode is always from Router, not from model output ---
# Even if the mock dict contains the wrong mode, the payload must use the Router's mode.
signal = make_signal(distress='moderate', support=['emotional_validation'], confidence=0.72)
comfort_decision = router.route(signal)

mock_wrong_mode = {
    'mode': 'guidance',     # model hallucinated wrong mode
    'distress_level': 'low',# model hallucinated wrong distress
    'tone_guidance': 'Warm and validating.',
    'framing_strategy': 'Lead with acknowledgement.',
    'response_constraints': ['No advice.']
}
strategy = ssa.mock_strategize(mock_wrong_mode, comfort_decision, signal)
ok = (
    strategy.mode == OperationalMode.COMFORT        # Router wins
    and strategy.distress_level == DistressLevel.MODERATE   # Signal Agent wins
)
print(f'T5 mode/distress always from Router/Signal : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   mode={strategy.mode.value} (Router said COMFORT, mock said guidance)')

T5 mode/distress always from Router/Signal : ✓
   mode=comfort (Router said COMFORT, mock said guidance)


In [8]:
# --- T6: safe fallback triggers on empty fields ---
signal = make_signal(distress='moderate', support=['emotional_validation'], confidence=0.72)
comfort_decision = router.route(signal)

mock_empty = {'tone_guidance': '', 'framing_strategy': '', 'response_constraints': []}
strategy = ssa.mock_strategize(mock_empty, comfort_decision, signal)
has_fallback = any('FALLBACK' in c for c in strategy.response_constraints)
ok = has_fallback and strategy.mode == OperationalMode.COMFORT
print(f'T6 empty fields trigger fallback : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   constraints: {strategy.response_constraints}')

[SupportStrategyAgent] FALLBACK triggered: Model returned empty tone_guidance or framing_strategy.
T6 empty fields trigger fallback : ✓
   constraints: ['No advice unless explicitly requested.', 'No solution-framing.', 'Keep responses short and human.', '[FALLBACK] Model returned empty tone_guidance or framing_strategy.']


In [9]:
# --- T7: JSON extraction handles markdown fences ---
fenced = '```json\n{"tone_guidance": "Warm.", "framing_strategy": "Validate first.", "response_constraints": ["No advice."]}\n```'
parsed = ssa._extract_json(fenced)
ok = parsed.get('tone_guidance') == 'Warm.'
print(f'T7 JSON extraction from markdown fence : {chr(10003) if ok else chr(10007)+" FAIL"}')
print(f'   tone_guidance={parsed.get("tone_guidance")!r}')

T7 JSON extraction from markdown fence : ✓
   tone_guidance='Warm.'


## Part B — Live model call (GPU required)
Run only when the model is available. Reuses the model loaded in Step 3 notebook if the kernel is still warm.


In [10]:
import bitsandbytes as bnb
print(bnb.__version__)

0.45.5


In [11]:
# --- B1: Live COMFORT strategy ---
signal = make_signal(
    distress='moderate',
    support=['emotional_validation', 'grounding_stabilization'],
    confidence=0.74
)
comfort_decision = router.route(signal)
print(f'Router decision: {comfort_decision.mode.value}')
print()
strategy = ssa.strategize(comfort_decision, signal)
print(f'mode              : {strategy.mode.value}')
print(f'distress_level    : {strategy.distress_level.value}')
print(f'tone_guidance     : {strategy.tone_guidance}')
print(f'framing_strategy  : {strategy.framing_strategy}')
print(f'constraints ({len(strategy.response_constraints)}):')
for c in strategy.response_constraints:
    print(f'  - {c}')

Router decision: comfort



C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SupportStrategyAgent] Loading model: Qwen/Qwen2.5-3B-Instruct


C:\Users\dalet\anaconda3\envs\nikko\Lib\site-packages\accelerate\utils\modeling.py:1365: UserWarning: Current model requires 50332032 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(
Loading checkpoint shards: 100%|█████████████████| 2/2 [00:06<00:00,  3.15s/it]


[SupportStrategyAgent] Model loaded in 10.1s.
mode              : comfort
distress_level    : moderate
tone_guidance     : Use a warm, present, and validating tone to acknowledge the user's emotions without overwhelming them.
framing_strategy  : Start by acknowledging the user's sadness and low mood, then gently offer support and reassurance without prescribing solutions.
constraints (3):
  - No directive advice
  - Avoid clinical authority language
  - Encourage seeking professional help if needed


In [12]:
# --- B2: Live GUIDANCE strategy ---
signal = make_signal(
    distress='low',
    support=['psychoeducation', 'normalization'],
    behavioral=['help_seeking_behavior'],
    confidence=0.82
)
guidance_decision = router.route(signal)
print(f'Router decision: {guidance_decision.mode.value}')
print()
strategy = ssa.strategize(guidance_decision, signal)
print(f'mode              : {strategy.mode.value}')
print(f'tone_guidance     : {strategy.tone_guidance}')
print(f'framing_strategy  : {strategy.framing_strategy}')
print(f'constraints ({len(strategy.response_constraints)}):')
for c in strategy.response_constraints:
    print(f'  - {c}')

Router decision: guidance

mode              : guidance
tone_guidance     : calm, informative, grounded. Emphasize empathy and provide clear, non-prescriptive information.
framing_strategy  : Lead with understanding and validate the user's feelings before offering information. Frame the content as something that might be helpful rather than a definitive guide.
constraints (5):
  - no directive advice
  - no clinical authority language
  - no evidence injection unless it arises naturally from a validation need
  - encourage the user to verify with a professional
  - autonomy language required
